# Data Preparation and Cleaning

## Import Libraries


In [10]:
import requests
import pandas as pd
import os

## Load Census API Data (2012)

In [11]:
# Ensure data directory exists
os.makedirs('../data', exist_ok=True)

CENSUS_API_KEY = "c162f3620fb054f8b97d67a3ef165e8a8417a185"
csv_path_2012 = "../data/census_2012_sba.csv"

if os.path.exists(csv_path_2012) :
    print(f"File '{csv_path_2012}' exists. Skipping API data load.")
    df_2012 = pd.read_csv(csv_path_2012)
    print(f"Loaded 2012 dataframe from '{csv_path_2012}' with {len(df_2012)} rows and {len(df_2012.columns)} columns.")
else:
    print("Fetching data for 2012...")
    url_2012 = "https://api.census.gov/data/2012/sbo/cs"
    vars_2012 = "NAME,NAICS2012,NAICS2012_LABEL,SEX,SEX_LABEL,ETH_GROUP,ETH_GROUP_LABEL,RACE_GROUP,RACE_GROUP_LABEL,VET_GROUP,VET_GROUP_LABEL,FIRMALL,RCPALL,EMP"
    params_2012 = {"get": vars_2012, "for": "state:*"}
    if CENSUS_API_KEY:
        params_2012["key"] = CENSUS_API_KEY
    response_2012 = requests.get(url_2012, params=params_2012)
    response_2012.raise_for_status()
    data_2012 = response_2012.json()
    df_2012 = pd.DataFrame(data_2012[1:], columns=data_2012[0])
    df_2012['YEAR'] = 2012
    df_2012.to_csv(csv_path_2012, index=False)
    print(f"Successfully fetched and saved {len(df_2012)} rows for 2012 to '{csv_path_2012}'.\n")



File '../data/census_2012_sba.csv' exists. Skipping API data load.


/tmp/ipython-input-495/1030065439.py:9: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  df_2012 = pd.read_csv(csv_path_2012)


Loaded 2012 dataframe from '../data/census_2012_sba.csv' with 2070597 rows and 16 columns.


## Load Census data for 2002 and 2007 (CSV)

In [12]:
import os
import requests
# Replace with your actual Dropbox direct download links for each file
sb2002_path = os.path.join('../data', 'census_2002_sba.csv')
sb2002_url = 'https://www.dropbox.com/scl/fi/4k2w0hvhl5clwsefl4lll/census_2002_sba.csv?rlkey=kvwjfifle9ov24d4vfsjhvuyi&st=2u8t41kq&dl=1'
sb2007_path = os.path.join('../data', 'census_2007_sba.csv')
sb2007_url = 'https://www.dropbox.com/scl/fi/6lw0l0gmzddcnvn0v6ebc/census_2007_sba.csv?rlkey=coiqvmv8i91z0lbhynk1aornr&st=iahz4mnu&dl=1'

if not os.path.exists(sb2002_path):
    print("Downloading census_2002_sba.csv ...")
    response = requests.get(sb2002_url)
    response.raise_for_status()
    with open(sb2002_path, 'wb') as f:
        f.write(response.content)
    print("Downloaded census_2002_sba.csv")
else:
    print("census_2002_sba.csv already exists.")

if not os.path.exists(sb2007_path):
    print("Downloading census_2007_sba.csv ...")
    response = requests.get(sb2007_url)
    response.raise_for_status()
    with open(sb2007_path, 'wb') as f:
        f.write(response.content)
    print("Downloaded census_2007_sba.csv.")
else:
    print("census_2007_sba.csv already exists.")

census_2002_sba.csv already exists.
census_2007_sba.csv already exists.


In [13]:
FRED_API_KEY = "175c981a568a2ea162c4d2a03cf1209c"

series_ids = ["FEDFUNDS", "MPRIME", "CPIAUCSL", "UNRATE"]
base_url = "https://api.stlouisfed.org/fred/series/observations"

all_series_df = []

for series_id in series_ids:
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "frequency": "m",
        "observation_start": "1987-01-01",
        "observation_end": "2014-12-31"
    }

    response = requests.get(base_url, params=params)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["observations"])[["date", "value"]]
    df.rename(columns={"value": series_id}, inplace=True)

    all_series_df.append(df)

merged_df = all_series_df[0]
for series_df in all_series_df[1:]:
    merged_df = pd.merge(merged_df, series_df, on="date", how="outer")

merged_df.to_csv("../data/fred_1987_2014.csv", index=False)
print("\nData saved to fred_1987_2014.csv")


Data saved to fred_1987_2014.csv


## Data Cleaning and Analysis

### Census Data Analysis and Cleaning

In [14]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)

data_dir = '../data'

# Load census data
census_2002_df = pd.read_csv(os.path.join(data_dir, 'census_2002_sba.csv'))
census_2007_df = pd.read_csv(os.path.join(data_dir, 'census_2007_sba.csv'))
census_2012_df = pd.read_csv(os.path.join(data_dir, 'census_2012_sba.csv'))

# Basic EDA for each DataFrame (no loops)
print("\n--- CENSUS 2002 ---")
print(census_2002_df.head())
print(census_2002_df.info())
print(census_2002_df.describe())

print("\n--- CENSUS 2007 ---")
print(census_2007_df.head())
print(census_2007_df.info())
print(census_2007_df.describe())

print("\n--- CENSUS 2012 ---")
print(census_2012_df.head())
print(census_2012_df.info())
print(census_2012_df.describe())

/tmp/ipython-input-495/1209054747.py:11: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  census_2012_df = pd.read_csv(os.path.join(data_dir, 'census_2012_sba.csv'))



--- CENSUS 2002 ---
   GEOTYPE  ST GEOGRAPHY  RACE_GROUP RACE_GROUP_MEANING  SEX  \
0        2   1   Alabama           0          All firms    0   
1        2   1   Alabama          30              White    0   
2        2   1   Alabama          30              White    1   
3        2   1   Alabama          30              White    2   
4        2   1   Alabama          30              White    3   

                  SEX_MEANING NAICS2002      NAICS2002_MEANING  FIRMALL  \
0                   All firms        00  Total for all sectors   309544   
1                   All firms        00  Total for all sectors   265110   
2                Female-owned        00  Total for all sectors    67224   
3                  Male-owned        00  Total for all sectors   170457   
4  Equally male-/female-owned        00  Total for all sectors    27429   

      RCPALL      EMP  YEAR  
0  266638167  1507494  2007  
1  115638632   783378  2007  
2   10750766    89569  2007  
3   98262696   632520  

#### Standardize Columns and Merge

In [15]:
# Map to state code
state_name_to_iso = {
    'Alabama': 'AL', 'Alaska': 'AK', 'Arizona': 'AZ', 'Arkansas': 'AR', 'California': 'CA',
    'Colorado': 'CO', 'Connecticut': 'CT', 'Delaware': 'DE', 'District of Columbia': 'DC', 'Florida': 'FL',
    'Georgia': 'GA', 'Hawaii': 'HI', 'Idaho': 'ID', 'Illinois': 'IL', 'Indiana': 'IN',
    'Iowa': 'IA', 'Kansas': 'KS', 'Kentucky': 'KY', 'Louisiana': 'LA', 'Maine': 'ME',
    'Maryland': 'MD', 'Massachusetts': 'MA', 'Michigan': 'MI', 'Minnesota': 'MN', 'Mississippi': 'MS',
    'Missouri': 'MO', 'Montana': 'MT', 'Nebraska': 'NE', 'Nevada': 'NV', 'New Hampshire': 'NH',
    'New Jersey': 'NJ', 'New Mexico': 'NM', 'New York': 'NY', 'North Carolina': 'NC', 'North Dakota': 'ND',
    'Ohio': 'OH', 'Oklahoma': 'OK', 'Oregon': 'OR', 'Pennsylvania': 'PA', 'Rhode Island': 'RI',
    'South Carolina': 'SC', 'South Dakota': 'SD', 'Tennessee': 'TN', 'Texas': 'TX', 'Utah': 'UT',

    'Vermont': 'VT', 'Virginia': 'VA', 'Washington': 'WA', 'West Virginia': 'WV', 'Wisconsin': 'WI', 'Wyoming': 'WY'
}

# Rename columns
census_2002_df_new = census_2002_df.rename(columns={
    'GEOGRAPHY': 'STATE_NAME',
    'NAICS2002': 'NAICS',
    'NAICS2002_MEANING': 'NAICS_NAME',
    'SEX_MEANING': 'SEX_NAME',
    'RACE_GROUP_MEANING': 'RACE_GROUP_NAME'
})

census_2007_df_new = census_2007_df.rename(columns={
    'GEOGRAPHY': 'STATE_NAME',
    'NAICS2007': 'NAICS',
    'NAICS2007_MEANING': 'NAICS_NAME',
    'SEX_MEANING': 'SEX_NAME',
    'ETH_GROUP_MEANING': 'ETH_GROUP_NAME',
    'RACE_GROUP_MEANING': 'RACE_GROUP_NAME'
})

census_2012_df_new = census_2012_df.rename(columns={
    'NAME': 'STATE_NAME',
    'NAICS2012': 'NAICS',
    'NAICS2012_LABEL': 'NAICS_NAME',
    'SEX_LABEL': 'SEX_NAME',
    'ETH_GROUP_LABEL': 'ETH_GROUP_NAME',
    'RACE_GROUP_LABEL': 'RACE_GROUP_NAME',
    'VET_GROUP_LABEL': 'VET_GROUP_NAME'
})

cols = [
    'YEAR',
    'STATE_CODE',
    'STATE_NAME',
    'NAICS',
    'NAICS_NAME',
    'SEX',
    'SEX_NAME',
    'ETH_GROUP',
    'ETH_GROUP_NAME',
    'RACE_GROUP',
    'RACE_GROUP_NAME',
    'VET_GROUP',
    'VET_GROUP_NAME',
    'FIRMALL',
    'RCPALL',
    'EMP',
]

census_2002_df_new['STATE_CODE'] = census_2002_df_new['STATE_NAME'].map(state_name_to_iso)
census_2007_df_new['STATE_CODE'] = census_2007_df_new['STATE_NAME'].map(state_name_to_iso)
census_2012_df_new['STATE_CODE'] = census_2012_df_new['STATE_NAME'].map(state_name_to_iso)

# Remove rows with missing or invalid state codes (summary/total rows)
for df in [census_2002_df_new, census_2007_df_new, census_2012_df_new]:
    df.dropna(subset=['STATE_CODE'], inplace=True)

# Add missing columns
for col in cols:
    if col not in census_2002_df_new.columns: census_2002_df_new[col] = pd.NA
    if col not in census_2007_df_new.columns: census_2007_df_new[col] = pd.NA
    if col not in census_2012_df_new.columns: census_2012_df_new[col] = pd.NA

# Subset and reorder columns
census_2002_df = census_2002_df_new[cols]
census_2007_df = census_2007_df_new[cols]
census_2012_df = census_2012_df_new[cols]

# Concatenate
merged_census_df = pd.concat([census_2002_df, census_2007_df, census_2012_df], ignore_index=True)

# Convert YEAR to categorical
merged_census_df['YEAR'] = merged_census_df['YEAR'].astype('category')

print('\nShape:')
print(merged_census_df.shape)

print('\nHead:')
print(merged_census_df.head())

print('\nStatistics:')
print(merged_census_df.describe(include='all'))

print('\nMissing values:')
print(merged_census_df.isna().sum())


Shape:
(3961277, 16)

Head:
   YEAR STATE_CODE STATE_NAME NAICS             NAICS_NAME  SEX  \
0  2007         AL    Alabama    00  Total for all sectors    0   
1  2007         AL    Alabama    00  Total for all sectors    0   
2  2007         AL    Alabama    00  Total for all sectors    1   
3  2007         AL    Alabama    00  Total for all sectors    2   
4  2007         AL    Alabama    00  Total for all sectors    3   

                     SEX_NAME ETH_GROUP ETH_GROUP_NAME  RACE_GROUP  \
0                   All firms      <NA>            NaN           0   
1                   All firms      <NA>            NaN          30   
2                Female-owned      <NA>            NaN          30   
3                  Male-owned      <NA>            NaN          30   
4  Equally male-/female-owned      <NA>            NaN          30   

  RACE_GROUP_NAME VET_GROUP VET_GROUP_NAME  FIRMALL     RCPALL      EMP  
0       All firms      <NA>            NaN   309544  266638167  1507494  

#### Standardize NAICS to NAICS_2012

In [16]:

# 1. Download/Load mapping files into small, temporary dataframes
url_12_07 = 'https://www.census.gov/naics/concordances/2012_to_2007_NAICS.xls'
url_07_02 = 'https://www.census.gov/naics/concordances/2007_to_2002_NAICS.xls'

df = merged_census_df.copy()

r1 = requests.get(url_12_07)
with open('../data/2012_to_2007.xls', 'wb') as f: f.write(r1.content)
r2 = requests.get(url_07_02)
with open('../data/2007_to_2002.xls', 'wb') as f: f.write(r2.content)

# Lookup Dictionaries
map_12_07 = pd.read_excel('../data/2012_to_2007.xls', skiprows=2, dtype=str)
map_07_02 = pd.read_excel('../data/2007_to_2002.xls', skiprows=2, dtype=str)

map_12_07 = map_12_07[['2012 NAICS Code', '2012 NAICS Title', '2007 NAICS Code']].drop_duplicates()
map_12_07.columns = ['2012_NAICS', '2012_TITLE', '2007_NAICS']

map_07_02 = map_07_02[['2007 NAICS Code', '2007 NAICS Title', '2002 NAICS Code']].drop_duplicates()
map_07_02.columns = ['2007_NAICS', '2007_TITLE', '2002_NAICS']

# Combine mappings
mapping_raw = pd.merge(map_12_07, map_07_02, on='2007_NAICS', how='outer')

# Identify 2002 Splits:
lookup_02 = mapping_raw.groupby('2002_NAICS').agg({
    '2012_NAICS': lambda x: ' & '.join(sorted(list(set(x.dropna())))),
    '2012_TITLE': lambda x: ' & '.join(sorted(list(set(x.dropna()))))
}).to_dict('index')

# Identify 2007 Splits
lookup_07 = mapping_raw.groupby('2007_NAICS').agg({
    '2012_NAICS': lambda x: ' & '.join(sorted(list(set(x.dropna())))),
    '2012_TITLE': lambda x: ' & '.join(sorted(list(set(x.dropna()))))
}).to_dict('index')

# Map individual codes
lookup_12 = {}
for code, vals in lookup_02.items():
    if ' & ' in vals['2012_NAICS']:
        for sub_code in vals['2012_NAICS'].split(' & '):
            lookup_12[sub_code] = vals


print("Fix NAICS format...")
df['NAICS'] = df['NAICS'].str.pad(6, side='right', fillchar='0')

# 2012 Lookup
mask_12 = df['YEAR'] == 2012
df.loc[mask_12, 'NAICS_2012_REV'] = df.loc[mask_12, 'NAICS'].map(lambda x: lookup_12.get(x, {}).get('2012_NAICS', x))
df.loc[mask_12, 'NAICS_2012_REV_NAME'] = df.loc[mask_12, 'NAICS'].map(lambda x: lookup_12.get(x, {}).get('2012_TITLE', None))

# 2007 Lookup
mask_07 = df['YEAR'] == 2007
df.loc[mask_07, 'NAICS_2012_REV'] = df.loc[mask_07, 'NAICS'].map(lambda x: lookup_07.get(x, {}).get('2012_NAICS', x))
df.loc[mask_07, 'NAICS_2012_REV_NAME'] = df.loc[mask_07, 'NAICS'].map(lambda x: lookup_07.get(x, {}).get('2012_TITLE', None))

# 2002 Lookup
mask_02 = df['YEAR'] == 2002
df.loc[mask_02, 'NAICS_2012_REV'] = df.loc[mask_02, 'NAICS'].map(lambda x: lookup_02.get(x, {}).get('2012_NAICS', x))
df.loc[mask_02, 'NAICS_2012_REV_NAME'] = df.loc[mask_02, 'NAICS'].map(lambda x: lookup_02.get(x, {}).get('2012_TITLE', None))

# Fill NA values with original names
df['NAICS_2012_REV_NAME'] = df['NAICS_2012_REV_NAME'].fillna(df['NAICS_NAME'])


group_cols = [
    'YEAR', 'STATE_CODE', 'STATE_NAME', 'NAICS', 'NAICS_NAME',
    'NAICS_2012_REV', 'NAICS_2012_REV_NAME',
    'SEX', 'SEX_NAME', 'ETH_GROUP', 'ETH_GROUP_NAME',
    'RACE_GROUP', 'RACE_GROUP_NAME', 'VET_GROUP', 'VET_GROUP_NAME'
]

print("Aggregating consolidated data...")
for c in ['STATE_CODE', 'SEX_NAME', 'ETH_GROUP_NAME', 'RACE_GROUP_NAME']:
    df[c] = df[c].astype('category')

df_consolidated = df.groupby(group_cols, dropna=False, observed=True, sort=False).agg({
    'FIRMALL': 'sum',
    'RCPALL': 'sum',
    'EMP': 'sum'
}).reset_index()

print("Saving consolidated data to CSV...")
df_consolidated.to_csv('../data/merged_census_sba.csv', index=False)

Fix NAICS format...
Aggregating consolidated data...
Saving consolidated data to CSV...


In [18]:
print('\nStatistics:')
df_consolidated.describe(include='all')


Statistics:


,YEAR,STATE_CODE,STATE_NAME,NAICS,NAICS_NAME,NAICS_2012_REV,NAICS_2012_REV_NAME,SEX,SEX_NAME,ETH_GROUP,ETH_GROUP_NAME,RACE_GROUP,RACE_GROUP_NAME,VET_GROUP,VET_GROUP_NAME,FIRMALL,RCPALL,EMP
count,3204889.0,3204889,3204889,3177532,3204889,3177532,3204889,3.204889e+06,3204889,3.181327e+06,3181327,3.204889e+06,3204889,1.634610e+06,1634610,3.204889e+06,3.204889e+06,3.204889e+06
unique,2.0,51,51,1713,1652,1535,2350,NaN,6,NaN,10,NaN,24,NaN,6,NaN,NaN,NaN
top,2012.0,CA,California,510000,Management of companies and enterprises,000000,Management of companies and enterprises,NaN,All firms,NaN,All firms,NaN,All firms,NaN,All firms,NaN,NaN,NaN
freq,1634610.0,86780,86780,17544,25888,17544,25888,NaN,2429729,NaN,2421629,NaN,1710125,NaN,1325827,NaN,NaN,NaN
mean,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.893344e+00,NaN,1.241586e+01,NaN,3.306287e+01,NaN,8.368584e+00,NaN,1.073305e+03,7.554545e+05,3.405578e+03
std,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.579437e+01,NaN,2.634409e+01,NaN,3.827851e+01,NaN,2.510228e+01,NaN,1.625657e+04,1.250005e+07,4.639394e+04
min,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,0.000000e+00,0.000000e+00
25%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,0.000000e+00,0.000000e+00
50%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN,1.000000e+00,NaN,0.000000e+00,NaN,1.000000e+00,NaN,3.000000e+00,0.000000e+00,0.000000e+00
75%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000e+00,NaN,1.000000e+00,NaN,6.700000e+01,NaN,1.000000e+00,NaN,6.400000e+01,5.249300e+04,5.840000e+02


In [17]:


print('\nMissing values:')
df_consolidated.isna().sum()


Statistics:

Missing values:


,0
YEAR,0
STATE_CODE,0
STATE_NAME,0
NAICS,27357
NAICS_NAME,0
NAICS_2012_REV,27357
NAICS_2012_REV_NAME,0
SEX,0
SEX_NAME,0
ETH_GROUP,23562
